# Figure 3H | Manual striatal subregions and normalized anatomical axes

Definition of three striatal subregions and construction of sample-specific anatomical coordinates used to compare spatial transcriptional patterns across tissue sections.

The workflow has two components:

1. **Manual striatal subregions**: dorsal striatum (`d`), nucleus accumbens (`n`), and olfactory tubercle (`o`) are provided by the manually curated `region_str_manual` annotation.
2. **Normalized anatomical axes**: for each sample and region, two anatomical reference contours are drawn manually. Each cell is assigned a relative position from 0 to 100 according to its shortest distance to the two contours.

| Region | x axis | y axis |
|---|---|---|
| Dorsal striatum (`d`) | anterior 0 → posterior 100 | dorsal 0 → ventral 100 |
| NAcc (`n`) | anterior 0 → posterior 100 | dorsal 0 → ventral 100 |
| OT (`o`) | anterior 0 → posterior 100 | not used |

The source notebooks implemented x and y separately using identical distance-ratio logic. They are consolidated here into one workflow with explicit anatomical boundary names.

The normalized coordinate is

**axis = 100 × d0 / (d0 + d100)**

where `d0` and `d100` are the shortest Euclidean distances from a cell to the reference contours assigned to the 0 and 100 ends of the corresponding anatomical axis.


## Required input

The input h5ad must contain:

- `obs['sample']`
- `obs['condition']` (used only for optional grouped visualization)
- `obs['region_str_manual']`, with `d`, `n`, and `o` denoting dorsal striatum, NAcc, and OT
- `obsm['position']` containing two-dimensional spatial coordinates

If `obsm['position']` is absent but `obs['x']` and `obs['y']` are available, the notebook constructs `position` automatically.

The original source notebooks also loaded several MSN-refinement objects to propagate cell-type annotations. Those annotations are not required to define the anatomical axes and are therefore not reproduced here.


## Imports and configuration


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from IPython.display import IFrame, display

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Paths: edit for the local repository/data layout.
# ---------------------------------------------------------------------
DATA_ROOT = Path("data")
OUT_DIR = Path("outputs/Fig3H")
FIG_DIR = OUT_DIR / "figures"
PICKER_DIR = OUT_DIR / "boundary_pickers"
BOUNDARY_DIR = OUT_DIR / "boundary_json"

# Source file used by the original x/y-axis notebooks.
INPUT_H5AD = DATA_ROOT / "hd_cut_cortex/adata_with_region_str_manual.h5ad"

# Final object containing both x and y anatomical coordinates.
OUTPUT_H5AD = OUT_DIR / "192genes_str_axes_xy.h5ad"

# Optional directories containing the original manually drawn JSON files.
# The loader below recognizes the legacy VM/DL JSON format.
LEGACY_X_DIR = Path("vm_dl_axis_picker")
LEGACY_Y_DIR = Path("vm_dl_axis_picker_y")

FIG_DIR.mkdir(parents=True, exist_ok=True)
PICKER_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

# Spatial calibration used only for scale bars.
UM_PER_COORDINATE = 0.16

REGION_NAME = {
    "d": "Dorsal striatum",
    "n": "NAcc",
    "o": "OT",
}

REGION_COLORS = {
    "d": "#4C78A8",
    "n": "#F58518",
    "o": "#54A24B",
}

# Final manuscript axis definitions.
AXIS_SPECS = {
    "x": {
        "regions": ["d", "n", "o"],
        "zero_name": "Anterior",
        "hundred_name": "Posterior",
        "obs_key": "region_manual_axis_x_0_100",
    },
    "y": {
        "regions": ["d", "n"],
        "zero_name": "Dorsal",
        "hundred_name": "Ventral",
        "obs_key": "region_manual_axis_y_0_100",
    },
}


## 1. Load the manually annotated striatal regions

The anatomical region annotation is treated as an upstream manual annotation. Only cells assigned to dorsal striatum (`d`), NAcc (`n`), or OT (`o`) are retained for the axis workflow.


In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

required_obs = {"sample", "region_str_manual"}
missing_obs = required_obs - set(adata.obs.columns)
if missing_obs:
    raise KeyError(f"Missing required obs columns: {sorted(missing_obs)}")

if "position" not in adata.obsm:
    if {"x", "y"}.issubset(adata.obs.columns):
        adata.obsm["position"] = adata.obs[["x", "y"]].to_numpy()
    else:
        raise KeyError(
            "The input requires obsm['position'] or obs['x']/obs['y']."
        )

adata = adata[
    adata.obs["region_str_manual"].astype(str).isin(["d", "n", "o"])
].copy()

print(adata)
print(
    pd.crosstab(
        adata.obs["sample"],
        adata.obs["region_str_manual"],
    )
)


## 2. Figure 3H schematic | Manual Dorsal / NAcc / OT regions

This diagnostic displays the manually defined subregions for an example section. Change `EXAMPLE_SAMPLE` to the section used in the final Figure 3H schematic.


In [ ]:
EXAMPLE_SAMPLE = str(
    sorted(adata.obs["sample"].astype(str).unique())[0]
)

def plot_manual_subregions(
    adata,
    sample,
    save_path=None,
    point_size=5,
):
    temp = adata[
        adata.obs["sample"].astype(str) == str(sample)
    ].copy()

    coords = np.asarray(temp.obsm["position"])[:, :2]
    labels = temp.obs["region_str_manual"].astype(str).to_numpy()

    fig, ax = plt.subplots(figsize=(6, 6))

    # Background cells are not present after the striatal-region subset;
    # plot each manual compartment explicitly.
    for region in ["d", "n", "o"]:
        mask = labels == region
        if not np.any(mask):
            continue

        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            color=REGION_COLORS[region],
            edgecolors="none",
            rasterized=True,
            label=REGION_NAME[region],
        )

    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_axis_off()
    ax.legend(frameon=False, markerscale=3)

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()


plot_manual_subregions(
    adata,
    EXAMPLE_SAMPLE,
    save_path=FIG_DIR / "Fig3H_manual_striatal_subregions.pdf",
)


# Part I | Draw sample- and region-specific reference contours

The source notebooks generated an interactive HTML canvas for every sample × region combination and manually clicked two reference contours.

The release version uses anatomically explicit names:

- **x axis:** anterior boundary = 0, posterior boundary = 100
- **y axis:** dorsal boundary = 0, ventral boundary = 100

OT is assigned only an x axis in the final analysis.

Each picker writes or expects a JSON file containing the two reference polylines. Existing JSON files produced by the original notebooks remain usable because the loader accepts the legacy keys `VM_curve_points` and `DL_curve_points`.


## 3. Interactive boundary-picker generator


In [ ]:
def make_boundary_picker_html(
    coords,
    output_html,
    axis_name,
    zero_name,
    hundred_name,
    width=700,
    height=900,
    point_size=1.5,
    invert_y=True,
    max_points=50000,
    random_state=0,
):
    """Create an interactive HTML page for manually drawing two contours."""

    coords = np.asarray(coords, dtype=float)

    if coords.ndim != 2 or coords.shape[1] < 2:
        raise ValueError("coords must be an N x 2 array.")

    if max_points is not None and coords.shape[0] > max_points:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(coords.shape[0], max_points, replace=False)
        show_coords = coords[idx]
    else:
        show_coords = coords

    xmin, xmax = float(coords[:, 0].min()), float(coords[:, 0].max())
    ymin, ymax = float(coords[:, 1].min()), float(coords[:, 1].max())

    pad_x = max((xmax - xmin) * 0.08, 1.0)
    pad_y = max((ymax - ymin) * 0.08, 1.0)

    xmin, xmax = xmin - pad_x, xmax + pad_x
    ymin, ymax = ymin - pad_y, ymax + pad_y

    points_json = json.dumps(show_coords.tolist())

    html = f'''
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
body {{ font-family: Arial, sans-serif; }}
canvas {{ border: 1px solid #aaa; cursor: crosshair; }}
button {{ margin: 4px; padding: 6px 10px; }}
textarea {{ font-family: monospace; font-size: 12px; }}
</style>
</head>
<body>

<h3>{axis_name}-axis boundary picker</h3>

<p>
Red: {zero_name} boundary = 0<br>
Blue: {hundred_name} boundary = 100
</p>

<button onclick="setMode('B0')">{zero_name} = 0</button>
<button onclick="setMode('B100')">{hundred_name} = 100</button>
<button onclick="undo()">Undo</button>
<button onclick="clearCurrent()">Clear current boundary</button>
<button onclick="showJSON()">Show JSON</button>

<br><br>
<canvas id="canvas" width="{width}" height="{height}"></canvas>
<div id="info"></div>
<br>
<textarea id="jsonBox" style="width:{width}px;height:240px;"></textarea>

<script>
const canvas = document.getElementById("canvas");
const ctx = canvas.getContext("2d");
const W = canvas.width, H = canvas.height;
const coords = {points_json};
const xmin={xmin}, xmax={xmax}, ymin={ymin}, ymax={ymax};
const invertY={str(invert_y).lower()}, pointSize={point_size};

let mode="B0";
let B0=[];
let B100=[];

function xToCanvas(x) {{
    return (x-xmin)/(xmax-xmin)*W;
}}

function yToCanvas(y) {{
    return invertY
        ? (y-ymin)/(ymax-ymin)*H
        : H-(y-ymin)/(ymax-ymin)*H;
}}

function canvasToX(cx) {{
    return xmin + cx/W*(xmax-xmin);
}}

function canvasToY(cy) {{
    return invertY
        ? ymin + cy/H*(ymax-ymin)
        : ymin + (H-cy)/H*(ymax-ymin);
}}

function setMode(m) {{
    mode=m;
    draw();
}}

function undo() {{
    if(mode==="B0" && B0.length>0) B0.pop();
    if(mode==="B100" && B100.length>0) B100.pop();
    draw();
}}

function clearCurrent() {{
    if(mode==="B0") B0=[];
    if(mode==="B100") B100=[];
    draw();
}}

function drawPolyline(points, color) {{
    if(points.length===0) return;

    ctx.strokeStyle=color;
    ctx.fillStyle=color;
    ctx.lineWidth=3;

    ctx.beginPath();

    for(let i=0; i<points.length; i++) {{
        const cx=xToCanvas(points[i][0]);
        const cy=yToCanvas(points[i][1]);

        if(i===0) ctx.moveTo(cx,cy);
        else ctx.lineTo(cx,cy);
    }}

    ctx.stroke();

    for(let i=0; i<points.length; i++) {{
        const cx=xToCanvas(points[i][0]);
        const cy=yToCanvas(points[i][1]);

        ctx.beginPath();
        ctx.arc(cx,cy,4,0,2*Math.PI);
        ctx.fill();
    }}
}}

function draw() {{
    ctx.clearRect(0,0,W,H);
    ctx.fillStyle="white";
    ctx.fillRect(0,0,W,H);

    ctx.fillStyle="#bbbbbb";

    for(const p of coords) {{
        const cx=xToCanvas(p[0]);
        const cy=yToCanvas(p[1]);

        ctx.beginPath();
        ctx.arc(cx,cy,pointSize,0,2*Math.PI);
        ctx.fill();
    }}

    drawPolyline(B0,"red");
    drawPolyline(B100,"blue");

    document.getElementById("info").innerHTML =
        "Mode: "+mode+
        " | boundary 0 points: "+B0.length+
        " | boundary 100 points: "+B100.length;
}}

canvas.addEventListener("click", function(event) {{
    const rect=canvas.getBoundingClientRect();
    const cx=event.clientX-rect.left;
    const cy=event.clientY-rect.top;

    const x=canvasToX(cx);
    const y=canvasToY(cy);

    if(mode==="B0") B0.push([x,y]);
    if(mode==="B100") B100.push([x,y]);

    draw();
}});

function getBoundaryJSON() {{
    return JSON.stringify(
        {{
            boundary_0_points:B0,
            boundary_100_points:B100,
            axis:"{axis_name}",
            boundary_0_name:"{zero_name}",
            boundary_100_name:"{hundred_name}",
            xmin:xmin,
            xmax:xmax,
            ymin:ymin,
            ymax:ymax,
            invert_y:invertY
        }},
        null,
        2
    );
}}

function showJSON() {{
    document.getElementById("jsonBox").value = getBoundaryJSON();
}}

draw();
</script>
</body>
</html>
'''

    Path(output_html).write_text(html, encoding="utf-8")

    return Path(output_html)


## 4. Generate pickers for all required sample × region × axis combinations

The output manifest records the expected HTML and JSON filenames. Open an HTML picker, draw both reference contours, click **Show JSON**, and save the displayed text to the corresponding `json_path`.

For an exact reproduction using already drawn source boundaries, place the original JSON files in the legacy directories configured above; the computation step searches those locations automatically.


In [ ]:
def build_boundary_manifest(adata):
    records = []

    samples = sorted(
        adata.obs["sample"].dropna().astype(str).unique()
    )

    for axis_name, spec in AXIS_SPECS.items():
        axis_picker_dir = PICKER_DIR / axis_name
        axis_json_dir = BOUNDARY_DIR / axis_name

        axis_picker_dir.mkdir(parents=True, exist_ok=True)
        axis_json_dir.mkdir(parents=True, exist_ok=True)

        for sample in samples:
            for region in spec["regions"]:
                mask = (
                    (adata.obs["sample"].astype(str) == sample)
                    & (
                        adata.obs["region_str_manual"].astype(str)
                        == region
                    )
                )

                coords = np.asarray(
                    adata.obsm["position"][mask.to_numpy(), :2]
                )

                if coords.shape[0] < 10:
                    print(
                        f"Skip {axis_name}: {sample}-{region}; "
                        "too few cells."
                    )
                    continue

                safe_sample = (
                    sample.replace("/", "_").replace(" ", "_")
                )

                html_path = (
                    axis_picker_dir
                    / f"{safe_sample}_{region}_{axis_name}_picker.html"
                )

                json_path = (
                    axis_json_dir
                    / f"{safe_sample}_{region}_{axis_name}_boundaries.json"
                )

                make_boundary_picker_html(
                    coords=coords,
                    output_html=html_path,
                    axis_name=axis_name,
                    zero_name=spec["zero_name"],
                    hundred_name=spec["hundred_name"],
                    random_state=0,
                )

                records.append({
                    "sample": sample,
                    "region": region,
                    "axis": axis_name,
                    "zero_name": spec["zero_name"],
                    "hundred_name": spec["hundred_name"],
                    "html_path": str(html_path),
                    "json_path": str(json_path),
                    "n_cells": int(coords.shape[0]),
                })

    return pd.DataFrame(records)


boundary_manifest = build_boundary_manifest(adata)

boundary_manifest.to_csv(
    OUT_DIR / "boundary_picker_manifest.csv",
    index=False,
)

boundary_manifest


### Optional: open one picker inside Jupyter


In [ ]:
# Example:
#
# row_id = 0
# display(
#     IFrame(
#         boundary_manifest.loc[row_id, "html_path"],
#         width=760,
#         height=1250,
#     )
# )
# boundary_manifest.loc[row_id]


### Optional: save copied JSON text from a picker

This helper is useful when working interactively in Jupyter: paste the JSON string copied from the HTML picker and specify the corresponding row in `boundary_manifest`.


In [ ]:
def save_boundary_json(boundary_json_text, manifest, row_id):
    boundary_data = json.loads(boundary_json_text)

    json_path = Path(
        manifest.loc[row_id, "json_path"]
    )
    json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(json_path, "w") as f:
        json.dump(boundary_data, f, indent=2)

    print("Saved:", json_path)


# Example:
#
# boundary_json_text = r'''
# {
#   "boundary_0_points": [...],
#   "boundary_100_points": [...]
# }
# '''
#
# save_boundary_json(
#     boundary_json_text,
#     boundary_manifest,
#     row_id=0,
# )


# Part II | Convert reference contours to 0–100 anatomical coordinates

For each cell, the two manually drawn polylines are densely sampled. A KD-tree is used to calculate the shortest Euclidean distance from the cell to each contour.

The resulting value is dimensionless and depends on the *relative* distance to the two reference contours rather than on tissue size. Therefore the coordinate can be compared across sections with different shape, orientation, or scale.


## 5. Distance-ratio functions


In [ ]:
def sample_polyline(points, n_per_segment=100):
    """Densely sample a manually drawn polyline."""

    points = np.asarray(points, dtype=float)

    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("Polyline points must be an N x 2 array.")

    if len(points) < 2:
        raise ValueError("Each boundary requires at least two points.")

    sampled = []

    for i in range(len(points) - 1):
        p1 = points[i]
        p2 = points[i + 1]

        t = np.linspace(
            0,
            1,
            n_per_segment,
            endpoint=False,
        )

        segment = (
            p1[None, :] * (1 - t[:, None])
            + p2[None, :] * t[:, None]
        )

        sampled.append(segment)

    sampled.append(points[-1][None, :])

    return np.vstack(sampled)


def compute_axis_between_boundaries(
    coords,
    boundary_0_points,
    boundary_100_points,
    n_per_segment=100,
    eps=1e-8,
):
    """
    Compute a normalized anatomical coordinate.

    axis = 100 * d0 / (d0 + d100)

    d0:
        shortest distance to the reference contour assigned 0.

    d100:
        shortest distance to the reference contour assigned 100.
    """

    coords = np.asarray(coords, dtype=float)

    boundary_0 = sample_polyline(
        boundary_0_points,
        n_per_segment=n_per_segment,
    )

    boundary_100 = sample_polyline(
        boundary_100_points,
        n_per_segment=n_per_segment,
    )

    d0, _ = cKDTree(boundary_0).query(coords)
    d100, _ = cKDTree(boundary_100).query(coords)

    score = (
        d0
        / (d0 + d100 + eps)
        * 100.0
    )

    return score


## 6. Boundary JSON loader

New release JSON files use `boundary_0_points` and `boundary_100_points`.

For backward compatibility, the loader also recognizes the source notebooks' legacy names:

- `VM_curve_points` → boundary 0
- `DL_curve_points` → boundary 100

Because the old variable names did not encode anatomical meaning, verify the polarity of legacy files before using them in the final manuscript analysis.


In [ ]:
def read_boundary_points(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    if (
        "boundary_0_points" in data
        and "boundary_100_points" in data
    ):
        p0 = data["boundary_0_points"]
        p100 = data["boundary_100_points"]

    elif (
        "VM_curve_points" in data
        and "DL_curve_points" in data
    ):
        p0 = data["VM_curve_points"]
        p100 = data["DL_curve_points"]

    else:
        raise KeyError(
            f"{json_path} does not contain recognized "
            "boundary point keys."
        )

    return (
        np.asarray(p0, dtype=float),
        np.asarray(p100, dtype=float),
    )


def find_boundary_json(row):
    """Find a new-format JSON or the corresponding legacy source JSON."""

    new_path = Path(row["json_path"])

    if new_path.exists():
        return new_path

    sample = str(row["sample"]).replace("/", "_").replace(" ", "_")
    region = str(row["region"])
    axis_name = str(row["axis"])

    if axis_name == "x":
        legacy_candidates = [
            LEGACY_X_DIR / f"{sample}_{region}_curves.json",
        ]
    elif axis_name == "y":
        legacy_candidates = [
            LEGACY_Y_DIR / f"{sample}_{region}_curves_y.json",
        ]
    else:
        legacy_candidates = []

    for path in legacy_candidates:
        if path.exists():
            return path

    return None


## 7. Calculate x and y coordinates for every applicable cell

Final output columns:

- `region_manual_axis_x_0_100`: anterior 0 → posterior 100
- `region_manual_axis_y_0_100`: dorsal 0 → ventral 100

OT receives an x coordinate only; its y coordinate remains `NaN`.


In [ ]:
for spec in AXIS_SPECS.values():
    adata.obs[spec["obs_key"]] = np.nan


missing_boundaries = []

for _, row in boundary_manifest.iterrows():
    json_path = find_boundary_json(row)

    if json_path is None:
        missing_boundaries.append(
            (
                row["sample"],
                row["region"],
                row["axis"],
            )
        )
        continue

    boundary_0, boundary_100 = read_boundary_points(
        json_path
    )

    mask = (
        (adata.obs["sample"].astype(str) == str(row["sample"]))
        & (
            adata.obs["region_str_manual"].astype(str)
            == str(row["region"])
        )
    )

    coords = np.asarray(
        adata.obsm["position"][mask.to_numpy(), :2]
    )

    score = compute_axis_between_boundaries(
        coords,
        boundary_0,
        boundary_100,
        n_per_segment=100,
    )

    obs_key = AXIS_SPECS[row["axis"]]["obs_key"]

    adata.obs.loc[mask, obs_key] = score


if missing_boundaries:
    print(
        f"{len(missing_boundaries)} sample/region/axis combinations "
        "do not yet have boundary JSON files."
    )
    print("First missing entries:", missing_boundaries[:10])
else:
    print("All required anatomical coordinates were calculated.")


### Backward-compatible x-axis alias

Some earlier downstream notebooks used `region_manual_axis_0_100` for the x axis. The alias below preserves compatibility while keeping the explicit final column name.


In [ ]:
adata.obs["region_manual_axis_0_100"] = (
    adata.obs["region_manual_axis_x_0_100"]
)


# Part III | Figure 3H visualization and quality control


## 8. Overlay manual boundaries and the resulting axis

Use this diagnostic to confirm that the 0 and 100 contours have the intended anatomical polarity before downstream gene-gradient analysis.


In [ ]:
def plot_axis_with_boundaries(
    adata,
    sample,
    region,
    axis_name,
    save_path=None,
    point_size=8,
):
    spec = AXIS_SPECS[axis_name]
    obs_key = spec["obs_key"]

    row_match = boundary_manifest[
        (boundary_manifest["sample"].astype(str) == str(sample))
        & (boundary_manifest["region"].astype(str) == str(region))
        & (boundary_manifest["axis"] == axis_name)
    ]

    if len(row_match) != 1:
        raise ValueError(
            "Expected exactly one manifest row for "
            f"{sample}, {region}, {axis_name}."
        )

    row = row_match.iloc[0]
    json_path = find_boundary_json(row)

    if json_path is None:
        raise FileNotFoundError(
            "No boundary JSON was found for this example."
        )

    b0, b100 = read_boundary_points(json_path)

    mask = (
        (adata.obs["sample"].astype(str) == str(sample))
        & (
            adata.obs["region_str_manual"].astype(str)
            == str(region)
        )
    )

    temp = adata[mask].copy()
    coords = np.asarray(temp.obsm["position"])[:, :2]
    values = temp.obs[obs_key].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(6, 6))

    scatter = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=values,
        s=point_size,
        cmap="Spectral_r",
        vmin=0,
        vmax=100,
        edgecolors="none",
        rasterized=True,
    )

    ax.plot(
        b0[:, 0],
        b0[:, 1],
        color="red",
        linewidth=2,
        label=f"{spec['zero_name']} = 0",
    )

    ax.plot(
        b100[:, 0],
        b100[:, 1],
        color="blue",
        linewidth=2,
        label=f"{spec['hundred_name']} = 100",
    )

    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_axis_off()
    ax.legend(frameon=False)

    cbar = fig.colorbar(
        scatter,
        ax=ax,
        fraction=0.035,
        pad=0.02,
    )

    cbar.set_label(
        f"{axis_name}-axis position (0–100)"
    )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()


In [ ]:
# Example after boundary JSON files have been supplied:
#
# plot_axis_with_boundaries(
#     adata,
#     sample=EXAMPLE_SAMPLE,
#     region="d",
#     axis_name="x",
#     save_path=FIG_DIR / "Fig3H_example_dorsal_x_axis.pdf",
# )
#
# plot_axis_with_boundaries(
#     adata,
#     sample=EXAMPLE_SAMPLE,
#     region="d",
#     axis_name="y",
#     save_path=FIG_DIR / "Fig3H_example_dorsal_y_axis.pdf",
# )


## 9. Axis-only map with a 500-μm scale bar

The spatial calibration (`1 coordinate unit = 0.16 μm`) is used only for the physical scale bar. The anatomical axis itself is dimensionless.


In [ ]:
def plot_axis_map_with_scalebar(
    adata,
    sample,
    region,
    axis_name,
    save_path=None,
    point_size=20,
    scale_bar_um=500,
):
    spec = AXIS_SPECS[axis_name]
    obs_key = spec["obs_key"]

    mask = (
        (adata.obs["sample"].astype(str) == str(sample))
        & (
            adata.obs["region_str_manual"].astype(str)
            == str(region)
        )
    )

    temp = adata[mask].copy()

    if temp.n_obs == 0:
        raise ValueError("No cells found for the requested sample/region.")

    coords = np.asarray(temp.obsm["position"])[:, :2]
    x = coords[:, 0]
    y = coords[:, 1]
    values = temp.obs[obs_key].to_numpy(dtype=float)

    width = max((x.max() - x.min()) / 3000, 3)
    height = max((y.max() - y.min()) / 3000, 3)

    fig, ax = plt.subplots(
        figsize=(width, height),
        dpi=80,
    )

    scatter = ax.scatter(
        x,
        y,
        c=values,
        s=point_size,
        cmap="Spectral_r",
        vmin=0,
        vmax=100,
        linewidths=0,
        rasterized=True,
    )

    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.invert_yaxis()

    # 500 μm scale bar using the source calibration.
    scale_bar_length = scale_bar_um / UM_PER_COORDINATE

    x_range = x.max() - x.min()
    y_range = y.max() - y.min()

    x_right = x.max() - 0.04 * x_range
    x_left = x_right - scale_bar_length
    y_bar = y.max() - 0.06 * y_range

    ax.plot(
        [x_left, x_right],
        [y_bar, y_bar],
        color="black",
        linewidth=4,
        solid_capstyle="butt",
        zorder=10,
    )

    ax.text(
        (x_left + x_right) / 2,
        y_bar - 0.025 * y_range,
        f"{scale_bar_um} μm",
        ha="center",
        va="bottom",
        fontsize=7,
        color="black",
        zorder=10,
    )

    cbar = fig.colorbar(
        scatter,
        ax=ax,
        fraction=0.035,
        pad=0.02,
    )

    cbar.set_label(
        f"{axis_name}-axis position (0–100)"
    )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()


## 10. Batch quality-control plots

These optional plots export the normalized anatomical coordinate for every available sample. They are intended to check orientation consistency across animals before downstream binning and gradient analysis.


In [ ]:
def export_axis_qc_maps(
    adata,
    axis_name,
    out_dir,
    point_size=5,
):
    spec = AXIS_SPECS[axis_name]
    obs_key = spec["obs_key"]

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for sample in sorted(
        adata.obs["sample"].astype(str).unique()
    ):
        for region in spec["regions"]:
            mask = (
                (adata.obs["sample"].astype(str) == sample)
                & (
                    adata.obs["region_str_manual"].astype(str)
                    == region
                )
            )

            temp = adata[mask].copy()

            if temp.n_obs == 0:
                continue

            coords = np.asarray(temp.obsm["position"])[:, :2]
            values = temp.obs[obs_key].to_numpy(dtype=float)

            fig, ax = plt.subplots(figsize=(5, 5))

            sca = ax.scatter(
                coords[:, 0],
                coords[:, 1],
                c=values,
                s=point_size,
                cmap="Spectral_r",
                vmin=0,
                vmax=100,
                linewidths=0,
                rasterized=True,
            )

            ax.set_aspect("equal")
            ax.invert_yaxis()
            ax.set_axis_off()
            ax.set_title(
                f"{sample} | {REGION_NAME[region]} | {axis_name}"
            )

            fig.colorbar(
                sca,
                ax=ax,
                fraction=0.035,
                pad=0.02,
            )

            fig.savefig(
                out_dir
                / f"{sample}_{region}_{axis_name}_axis.pdf",
                dpi=300,
                bbox_inches="tight",
            )

            plt.close(fig)


# Run after all boundary JSON files are available:
#
# export_axis_qc_maps(
#     adata,
#     "x",
#     FIG_DIR / "QC_x_axis",
# )
#
# export_axis_qc_maps(
#     adata,
#     "y",
#     FIG_DIR / "QC_y_axis",
# )


## 11. Save the anatomical-axis object


In [ ]:
adata.write_h5ad(OUTPUT_H5AD)

print("Saved:", OUTPUT_H5AD)

print(
    adata.obs[
        [
            "sample",
            "region_str_manual",
            "region_manual_axis_x_0_100",
            "region_manual_axis_y_0_100",
        ]
    ].head()
)
